**1. Import Libraries**

In [1]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

**2. Download NLTK Resources**

In [2]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

**3. Load the Dataset**

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
true = pd.read_csv('/content/drive/MyDrive/News_Dataset/True.csv')
fake = pd.read_csv('/content/drive/MyDrive/News_Dataset/Fake.csv')

In [5]:
fake.head()

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"


In [6]:
true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


**4. Add Labels**

In [7]:
fake["label"] = 0
true["label"] = 1

**5. Merge Both Datasets**

In [8]:
news = pd.concat([fake, true], ignore_index=True)

In [9]:
news.shape

(44898, 5)

**6. Shuffle Dataset**

In [10]:
news = news.sample(frac=1, random_state=42)

In [11]:
news.reset_index(drop=True, inplace=True)

**7. Keep Required Columns**

In [12]:
news = news[["text","label"]]

**8. Text Cleaning**

In [13]:
lemmatizer = WordNetLemmatizer()

In [14]:
stop_words = set(stopwords.words("english"))

In [15]:
corpus = []

for i in range(len(news)):

    review = re.sub('[^a-zA-Z]', ' ', news['text'][i])

    review = review.lower()

    words = review.split()

    words = [lemmatizer.lemmatize(word)
             for word in words
             if word not in stop_words]

    review = " ".join(words)

    corpus.append(review)

**9. Check Cleaned Text**

In [16]:
corpus[:3]

['st century wire say ben stein reputable professor pepperdine university also hollywood fame appearing tv show film ferris bueller day made provocative statement judge jeanine pirro show recently discussing halt imposed president trump executive order travel stein referred judgement th circuit court washington state coup tat executive branch constitution stein went call judge seattle political puppet judiciary political pawn watch interview complete statement note stark contrast rhetoric leftist medium pundit neglect note court ever blocked presidential order immigration past discus legal efficacy halt actual text executive order read trump news st century wire trump filessupport work subscribing becoming member wire tv',
 'washington reuters u president donald trump removed chief strategist steve bannon national security council wednesday reversing controversial decision early year give political adviser unprecedented role security discussion trump overhaul nsc confirmed white house 

**10. TF-IDF Feature Extraction**

In [17]:
tfidf = TfidfVectorizer(max_features=5000)

In [18]:
x = tfidf.fit_transform(corpus).toarray()

In [19]:
y = news["label"]

In [20]:
x.shape

(44898, 5000)

**11. Split Dataset**

In [21]:
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42
)

**12. Train Model**

In [22]:
model = LogisticRegression()

In [23]:
model.fit(x_train, y_train)

LogisticRegression()

**13. Prediction**

In [24]:
y_pred = model.predict(x_test)

**14. Accuracy**

In [25]:
accuracy_score(y_test, y_pred)

0.9841870824053452

**15. Classification Report**

In [26]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.99      0.98      0.98      4710
           1       0.98      0.99      0.98      4270

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



**16. Confusion Matrix**

In [27]:
print(confusion_matrix(y_test, y_pred))

[[4618   92]
 [  50 4220]]


**17. Test News**

In [28]:
sample = ["Aliens officially elected as the new leaders of Earth."]

In [29]:
sample = [re.sub('[^a-zA-Z]', ' ', text).lower() for text in sample]

In [30]:
sample = tfidf.transform(sample)

In [31]:
prediction = model.predict(sample)

In [32]:
if prediction[0] == 1:
    print("True News")
else:
    print("Fake News")

Fake News
